In [1]:
# descargamos los datos del ONI indicador principal de la NOAA (Oficina Nacional de Administración Oceánica y 
# Atmosférica (National Oceanic and Atmospheric Administration). Se trata de una prestigiosa agencia científica del 
# gobierno de los Estados Unidos —dependiente delDepartamento de Comercio— encargada de monitorear y estudiar las 
# condiciones del océano, la atmósfera y el espacio)
# para medir y clasificar las fases de El Niño-Oscilación del Sur (ENOS) 
# en el Océano Pacífico ecuatorial
import urllib.request

url = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"
urllib.request.urlretrieve(url, "oni_historico.txt")
print("Descargado correctamente")


Descargado correctamente


In [2]:
with open("oni_historico.txt") as f:
    for i in range(10):
        print(repr(f.readline()))

' SEAS  YR   TOTAL   ANOM\n'
'  DJF 1950  25.01  -1.32\n'
'  JFM 1950  25.36  -1.20\n'
'  FMA 1950  25.88  -1.12\n'
'  MAM 1950  26.24  -1.08\n'
'  AMJ 1950  26.35  -1.10\n'
'  MJJ 1950  26.32  -0.90\n'
'  JJA 1950  26.24  -0.62\n'
'  JAS 1950  25.94  -0.57\n'
'  ASO 1950  25.76  -0.56\n'


In [3]:
# SEAS -> ESTACIÓN DEL AÑO
# ANOM -> valor ONI 
# verano ( DJF)
# invierno ( JJA)
# comparamos el ONI de DJF de un año con las lluvias de JJA de ese mismo año.


In [4]:
# cargamos y filtramos el ONI de verano ( DJF)
import pandas as pd

df_oni = pd.read_csv("oni_historico.txt", sep=r"\s+")
df_oni.columns = ["temporada", "year", "total", "anom"]

oni_verano = df_oni[df_oni["temporada"] == "DJF"][["year", "anom"]].rename(columns={"anom": "oni_verano"})
print(f"Total de veranos con dato ONI: {len(oni_verano)}")
oni_verano.head()

Total de veranos con dato ONI: 77


,year,oni_verano
0,1950,-1.32
12,1951,-0.66
24,1952,0.52
36,1953,0.25
48,1954,0.41


In [5]:
df_precip = pd.read_excel("EC_series.xlsx")
df_precip.head()

,agno,mes,dia,valor
0,1974,1,1,0.0
1,1974,1,2,0.0
2,1974,1,3,0.0
3,1974,1,4,0.0
4,1974,1,5,0.0


In [6]:
df_precip = pd.read_excel("EC_series.xlsx")
print(df_precip.columns.tolist())
df_precip.head()

['agno', 'mes', 'dia', 'valor']


,agno,mes,dia,valor
0,1974,1,1,0.0
1,1974,1,2,0.0
2,1974,1,3,0.0
3,1974,1,4,0.0
4,1974,1,5,0.0


In [7]:
# Sumamos la precipitación de invierno (junio, julio, agosto) por año
precip_invierno = df_precip[df_precip["mes"].isin([6, 7, 8])].groupby("agno")["valor"].sum().reset_index()
precip_invierno = precip_invierno.rename(columns={"agno": "year", "valor": "precip_invierno_mm"})

print(f"Total de inviernos con datos: {len(precip_invierno)}")
precip_invierno.head()

Total de inviernos con datos: 46


,year,precip_invierno_mm
0,1974,307.0
1,1975,363.0
2,1976,90.0
3,1977,546.0
4,1978,410.0


In [8]:
dataset_nino = pd.merge(oni_verano, precip_invierno, on="year")
print(f"Años con ambos datos (ONI verano + lluvia invierno): {len(dataset_nino)}")

correlacion = dataset_nino[["oni_verano", "precip_invierno_mm"]].corr()
print(correlacion)

Años con ambos datos (ONI verano + lluvia invierno): 46
                    oni_verano  precip_invierno_mm
oni_verano            1.000000           -0.033738
precip_invierno_mm   -0.033738            1.000000


In [9]:
# correlación casi inexistente -0.03


In [10]:
# Contamos cuántos días con dato tiene cada invierno (debería ser ~92 días: jun+jul+ago)
dias_por_invierno = df_precip[df_precip["mes"].isin([6,7,8])].groupby("agno")["valor"].count().reset_index()
dias_por_invierno = dias_por_invierno.rename(columns={"agno":"year","valor":"dias_con_dato"})
dias_por_invierno.sort_values("dias_con_dato").head(10)

,year,dias_con_dato
0,1974,90
2,1976,90
3,1977,90
1,1975,91
9,1983,91
33,2007,91
43,2017,91
4,1978,92
7,1981,92
8,1982,92


In [11]:

# descartado que fuera un problema de datos faltantes.
# El siguiente paso es ver si la relación es más "categórica" que lineal
# comparando el promedio de lluvia según si el año fue Niño, Niña o Neutro:
def clasificar_enso(oni):
    if oni >= 0.5:
        return "Niño"
    elif oni <= -0.5:
        return "Niña"
    else:
        return "Neutro"

dataset_nino["categoria_enso"] = dataset_nino["oni_verano"].apply(clasificar_enso)

resumen = dataset_nino.groupby("categoria_enso")["precip_invierno_mm"].agg(["mean", "median", "count"])
print(resumen)

                      mean  median  count
categoria_enso                           
Neutro          324.435714  296.90     14
Niña            309.443750  280.35     16
Niño            322.150000  312.75     16


In [12]:
#Los promedios sí muestran una tendencia en la dirección esperada (Niño: 322mm > Niña: 309mm), pero la diferencia 
#es pequeña comparada con la variabilidad dentro 
#de cada grupo — por eso la correlación general salió casi plana. Antes de descartar la relación, hay dos cosas
#rigurosas que nos faltan hacer.

#1. Probar si la diferencia es estadísticamente significativa (no solo mirar el promedio a ojo — con solo 
#16 años por grupo, una diferencia de 13mm bien podría ser ruido):
from scipy import stats

lluvia_nino = dataset_nino[dataset_nino["categoria_enso"] == "Niño"]["precip_invierno_mm"]
lluvia_nina = dataset_nino[dataset_nino["categoria_enso"] == "Niña"]["precip_invierno_mm"]

t_stat, p_valor = stats.ttest_ind(lluvia_nino, lluvia_nina)
print(f"Valor p: {p_valor:.3f}")

Valor p: 0.806


In [14]:
# Cómo interpretarlo: si p_valor es menor a 0.05, la diferencia es estadísticamente significativa 
# (poco probable que sea solo azar). Si es mayor, no podemos afirmar con confianza que Niño y Niña sean
# realmente distintos con 
# estos datos.
#Un p de 0.806 es muy alto — estadísticamente, no hay evidencia de diferencia real entre años Niño y Niña en esta 
#estación específica. Toca ser honesto: con los datos actuales, la relación esperada no se confirma en Pichidegua.

#Antes de aceptar esto como conclusión final, hay dos causas más por descartar — y son metodológicamente
#importantes, no solo "intentar hasta que funcione":

#1. Ventana de lluvia demasiado estrecha. JJA (jun-ago) es el invierno astronómico, pero en Chile central la
#temporada de lluvias relevante suele ser más amplia (abril-septiembre). Si cortamos la temporada muy justo,
#podríamos estar perdiendo parte de la señal real:

In [15]:
precip_lluvias = df_precip[df_precip["mes"].isin([4,5,6,7,8,9])].groupby("agno")["valor"].sum().reset_index()
precip_lluvias = precip_lluvias.rename(columns={"agno": "year", "valor": "precip_lluvias_mm"})

dataset_nino2 = pd.merge(oni_verano, precip_lluvias, on="year")
print(dataset_nino2[["oni_verano", "precip_lluvias_mm"]].corr())

                   oni_verano  precip_lluvias_mm
oni_verano           1.000000          -0.051811
precip_lluvias_mm   -0.051811           1.000000


In [ ]:
# El desfase que usamos probablemente no es el correcto. Los análisis reales usan el ONI concurrente con la
# temporada de lluvias (mayo-agosto, la misma ventana que la lluvia), no el ONI del verano previo. Es decir: no es
# verano anticipa invierno", sino "el estado del Pacífico durante el invierno mismo" el que se correlaciona mejor
# tiene sentido físico, porque la teleconexión atmosférica opera mientras el fenómeno está activo, no como un
# pronóstico adelantado.

In [ ]:
# según un análisis reciente del CR2 (uno de los centros de investigación climática más serios de Chile), la
# correlación entre El Niño y la lluvia en Chile central se debilitó fuertemente después del año 2000 de sobre
# 0.7 entre 1970-1990, a solo ~0.2 después de 2000 (por causas aún no bien entendidas, asociadas a la "megasequía" 
# que afecta a Chile central desde hace más de una década). Como tus datos van de 1974 a 2020, estás promediando una
# época de relación fuerte con otra de relación débil/rota eso puede estar aplanando tu correlación general.

In [16]:
# 1. Usamos el ONI promedio de mayo-agosto (concurrente), no el de verano
oni_mjja = df_oni[df_oni["temporada"].isin(["MJJ","JJA"])].groupby("year")["anom"].mean().reset_index()
oni_mjja = oni_mjja.rename(columns={"anom": "oni_mjja"})

dataset_v2 = pd.merge(oni_mjja, precip_invierno, on="year")

# 2. Dividimos en dos períodos para ver si la relación se debilitó, igual que documenta la literatura
periodo1 = dataset_v2[dataset_v2["year"] < 2000]
periodo2 = dataset_v2[dataset_v2["year"] >= 2000]

print("1974-1999:", periodo1[["oni_mjja","precip_invierno_mm"]].corr().iloc[0,1])
print("2000-2020:", periodo2[["oni_mjja","precip_invierno_mm"]].corr().iloc[0,1])
print("Todo el período:", dataset_v2[["oni_mjja","precip_invierno_mm"]].corr().iloc[0,1])

1974-1999: 0.46860216181871667
2000-2020: -0.027616965013425466
Todo el período: 0.32178985549403105


In [ ]:
## 1974-1999: correlación de 0.47 (moderada-fuerte)  la relación El Niño-lluvia funcionaba bien
## 2000-2020: correlación de -0.03 (nula)  la relación se rompió
## Todo el período junto: 0.32  un promedio engañoso que esconde ambos comportamientos

In [ ]:
# el resultado es rico porque cuenta una historia real: algo cambió en el sistema climático
# no que la relación nunca existió.

In [ ]:
# el desfase que asumimos al principio (verano→invierno) estaba mal — la relación correcta es concurrente
# (MJJA↔MJJA), y eso también fue parte del aprendizaje: revisar la literatura antes de asumir un desfase
# temporal "porque suena lógico".